<a href="https://colab.research.google.com/github/sebastianatanasovici-wq/-notebook-/blob/main/pregunta_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from __future__ import annotations

from functools import wraps
from time import perf_counter

In [ ]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed
    return wrapper

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 45.9 MB/s eta 0:00:00


# **4)Donovan**

***Restricciones críticas:***

Balance de inventario en cada trimestre

***Interpretación:***

El sistema busca equilibrio entre:

contratar trabajadores

mantener inventario

***Sensibilidad:***

Si sube el costo de inventario → se produce más “justo a tiempo”

Si sube el salario → se reduce la cantidad de trabajadores

Si aumenta la demanda → aumenta la contratación

**Demanda**

T1: 4000

T2: 2000

T3: 3000

T4: 10000

**Trabajadores**

Cada empleado:

trabaja 3 trimestres

descansa 1 trimestre

produce 500 por trimestre

salario: 30000 al año

**Inventario:**

Inicial: 600

Costo: 30 por unidad por trimestre

In [ ]:
def create_ampl_instance(solver: str = 'highs'):
    return ampl_notebook(modules=[solver], license_uuid='default')


def extract_solution(ampl):
    return {
        "Descanso_T1": ampl.get_variable("x1").value(),
        "Descanso_T2": ampl.get_variable("x2").value(),
        "Descanso_T3": ampl.get_variable("x3").value(),
        "Descanso_T4": ampl.get_variable("x4").value(),
        "Inventario_T1": ampl.get_variable("I1").value(),
        "Inventario_T2": ampl.get_variable("I2").value(),
        "Inventario_T3": ampl.get_variable("I3").value(),
        "Inventario_T4": ampl.get_variable("I4").value(),
        "Costo_Total": ampl.get_objective("Costo").value(),
    }


@timed
def solve_donovan_problem():
    ampl = create_ampl_instance()

    ampl.eval(r'''
        # VARIABLES
        var x1 >= 0;
        var x2 >= 0;
        var x3 >= 0;
        var x4 >= 0;

        var I1 >= 0;
        var I2 >= 0;
        var I3 >= 0;
        var I4 >= 0;

        # FUNCION OBJETIVO
        minimize Costo:
            30000*(x1+x2+x3+x4)
            + 30*(I1+I2+I3+I4);

        # PRODUCCION POR TRIMESTRE

        # T1
        s1:
        600 + 500*(x2+x3+x4) - 4000 = I1;

        # T2
        s2:
        I1 + 500*(x1+x3+x4) - 2000 = I2;

        # T3
        s3:
        I2 + 500*(x1+x2+x4) - 3000 = I3;

        # T4
        s4:
        I3 + 500*(x1+x2+x3) - 10000 = I4;
    ''')

    ampl.option["solver"] = "highs"
    ampl.solve()

    return extract_solution(ampl)


resultado, tiempo = solve_donovan_problem()

print("=== RESULTADOS EJERCICIO 4 ===")
print(f"Solución -> {resultado}")
print(f"Tiempo   -> {tiempo:.6f} segundos")

Using default Community Edition License for Colab. Get yours at: https://ampl.com/ce
Licensed to AMPL Community Edition License for the AMPL Model Colaboratory (https://ampl.com/colab).
HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 506000
4 simplex iterations
0 barrier iterations
=== RESULTADOS EJERCICIO 4 ===
Solución -> {'Descanso_T1': 5.466666666666666, 'Descanso_T2': 6.8, 'Descanso_T3': 0.0, 'Descanso_T4': 0.0, 'Inventario_T1': 0.0, 'Inventario_T2': 733.3333333333334, 'Inventario_T3': 3866.666666666667, 'Inventario_T4': 0.0, 'Costo_Total': 506000.0}
Tiempo   -> 11.689375 segundos
